**© Copyright AIDENTIFY. All rights reserved.**

# Part 4 | Session 28: LangChain RAG 어플리케이션 구현

## 📋 학습 목표

1️⃣ LangChain의 Document Loaders로 다양한 형식의 문서를 로딩한다  
2️⃣ Text Splitters로 문서를 효과적으로 분할한다  
3️⃣ Embeddings와 VectorStore를 연동한다  
4️⃣ RetrievalQA 체인으로 RAG 어플리케이션을 구축한다  
5️⃣ 대화형 RAG (ConversationalRetrievalChain)를 구현한다  

---

### 🖥️ 실습 환경
- **GPU**: 불필요 (API 기반)
- **필수 패키지**: langchain, langchain-openai, langchain-community, chromadb
- **LLM**: OpenAI API 또는 Ollama 로컬 모델

## 1️⃣ 패키지 및 환경 설정

In [ ]:
# 📦 패키지 확인 및 환경 설정
import importlib
import os

packages = [
    "langchain",
    "langchain_openai",
    "langchain_community",
    "chromadb",
]

print("📦 패키지 버전 확인")
print("=" * 40)
for pkg_name in packages:
    try:
        pkg = importlib.import_module(pkg_name)
        version = getattr(pkg, "__version__", "installed")
        print(f"  ✅ {pkg_name}: {version}")
    except ImportError:
        print(f"  ❌ {pkg_name}: 설치 필요")
        print(f"     pip install {pkg_name}")

In [ ]:
# 🔑 API 키 설정
# 방법 1: 환경변수로 설정 (권장)
# os.environ["OPENAI_API_KEY"] = "sk-your-key-here"

# 방법 2: Ollama 로컬 모델 사용 (API 키 불필요)
USE_OLLAMA = True  # True면 Ollama, False면 OpenAI 사용

if USE_OLLAMA:
    print("🔧 Ollama 로컬 모델을 사용합니다.")
    print("   ollama serve 실행 후 진행해주세요.")
else:
    api_key = os.environ.get("OPENAI_API_KEY", "")
    if api_key:
        print(f"✅ OpenAI API 키 설정 완료 (키: ...{api_key[-4:]})")
    else:
        print("⚠️ OPENAI_API_KEY가 설정되지 않았습니다.")

## 2️⃣ LangChain Document Loaders

LangChain은 다양한 형식의 문서를 로딩하는 **Document Loader**를 제공합니다.

### 📁 주요 Document Loaders
| Loader | 지원 형식 | 설명 |
|--------|----------|------|
| `TextLoader` | .txt | 텍스트 파일 |
| `PyPDFLoader` | .pdf | PDF 파일 |
| `CSVLoader` | .csv | CSV 파일 |
| `WebBaseLoader` | URL | 웹 페이지 |
| `DirectoryLoader` | 폴더 | 폴더 내 전체 파일 |

In [ ]:
# 📄 실습용 텍스트 파일 생성
import tempfile
import os

# 임시 디렉토리에 샘플 파일 생성
sample_dir = tempfile.mkdtemp(prefix="rag_docs_")

documents_content = {
    "ai_history.txt": """인공지능의 역사와 발전

인공지능(AI)은 1956년 다트머스 회의에서 처음 학문 분야로 정립되었습니다.
초기 AI는 규칙 기반 전문가 시스템이 주류였으며, 체스 프로그램 등이 대표적입니다.

1980년대에는 AI 겨울이 찾아와 연구 투자가 크게 줄었습니다.
하지만 2012년 AlexNet이 이미지넷 대회에서 압도적인 성적을 거두며 딥러닝 시대가 열렸습니다.

2017년 구글이 트랜스포머 아키텍처를 발표하면서 자연어 처리 분야에 혁명이 일어났습니다.
BERT, GPT 등의 대규모 언어 모델이 등장하여 다양한 NLP 태스크에서 인간 수준의 성능을 달성했습니다.

2022년 ChatGPT가 출시되며 AI가 일반 대중에게 널리 알려지게 되었습니다.
현재 생성형 AI는 텍스트, 이미지, 코드, 음악 등 다양한 콘텐츠를 생성할 수 있습니다.""",

    "llm_training.txt": """대규모 언어 모델(LLM) 학습 방법

사전학습(Pretraining)은 대량의 텍스트 데이터로 언어의 패턴을 학습하는 단계입니다.
다음 토큰 예측(Next Token Prediction) 방식으로 수조 개의 토큰을 학습합니다.

파인튜닝(Fine-tuning)은 특정 작업에 맞게 모델을 추가 학습하는 과정입니다.
SFT(Supervised Fine-Tuning)는 질문-답변 쌍으로 지시 따르기 능력을 향상시킵니다.

LoRA(Low-Rank Adaptation)는 전체 파라미터 중 일부만 학습하여 효율성을 높입니다.
QLoRA는 4bit 양자화와 LoRA를 결합하여 소비자급 GPU에서도 학습이 가능합니다.

RLHF(Reinforcement Learning from Human Feedback)는 인간의 선호도를 학습에 반영합니다.
DPO(Direct Preference Optimization)는 RLHF를 단순화한 방법론입니다.""",

    "rag_guide.txt": """RAG(Retrieval-Augmented Generation) 가이드

RAG는 검색 증강 생성이라고 하며, LLM의 환각 문제를 해결하는 핵심 기술입니다.
외부 지식 베이스에서 관련 정보를 검색한 후 LLM에 컨텍스트로 제공합니다.

RAG 파이프라인의 주요 단계:
1. 문서 수집 및 로딩
2. 텍스트 청킹 (적절한 크기로 분할)
3. 임베딩 생성 (텍스트를 벡터로 변환)
4. 벡터 DB에 저장 (ChromaDB, FAISS 등)
5. 사용자 질문에 대해 유사 문서 검색
6. 검색된 문서와 함께 LLM에 질의

청킹 전략, 임베딩 모델 선택, 검색 알고리즘이 RAG의 성능을 결정합니다.
Advanced RAG 기법으로 HyDE, Reranking, Query Expansion 등이 있습니다."""
}

for filename, content in documents_content.items():
    filepath = os.path.join(sample_dir, filename)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

print(f"📁 샘플 파일 생성 완료: {sample_dir}")
for f in os.listdir(sample_dir):
    size = os.path.getsize(os.path.join(sample_dir, f))
    print(f"  📄 {f} ({size} bytes)")

In [ ]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader

# 📂 DirectoryLoader로 폴더 내 모든 텍스트 파일 로딩
print("📂 문서 로딩 시작")
print("=" * 50)

loader = DirectoryLoader(
    sample_dir,
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

documents = loader.load()

print(f"✅ {len(documents)}개 문서 로딩 완료")
for i, doc in enumerate(documents, 1):
    source = os.path.basename(doc.metadata.get('source', 'unknown'))
    print(f"  {i}. {source} ({len(doc.page_content)}자)")
    print(f"     미리보기: {doc.page_content[:80]}...")

## 3️⃣ Text Splitters로 문서 분할

LangChain의 다양한 Text Splitter를 활용하여 문서를 청킹합니다.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# ✂️ RecursiveCharacterTextSplitter 사용
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],
)

print("✂️ 문서 분할 (RecursiveCharacterTextSplitter)")
print(f"  📏 chunk_size: 300, chunk_overlap: 50")
print("=" * 50)

splits = text_splitter.split_documents(documents)

print(f"\n📊 결과:")
print(f"  원본 문서: {len(documents)}개")
print(f"  분할 청크: {len(splits)}개")

# 청크 미리보기
print(f"\n🔍 처음 5개 청크 미리보기:")
for i, split in enumerate(splits[:5]):
    source = os.path.basename(split.metadata.get('source', 'unknown'))
    print(f"  {i+1}. [{source}] ({len(split.page_content)}자)")
    print(f"     {split.page_content[:80]}...")

## 4️⃣ Embeddings & VectorStore 연동

LangChain의 Embeddings 인터페이스와 ChromaDB를 연동합니다.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 📐 임베딩 모델 설정
print("📐 임베딩 모델 로딩 중...")
embeddings = HuggingFaceEmbeddings(
    model_name="BM-K/KoSimCSE-roberta-multitask"   # 한국어 최적화 (옵션: all-MiniLM-L6-v2, paraphrase-multilingual-mpnet-base-v2),
    model_kwargs={"device": "cpu"},
)
print("✅ 임베딩 모델 로딩 완료")

# 🗃️ ChromaDB VectorStore 생성
print("\n🗃️ VectorStore 생성 중...")
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="langchain_rag_demo",
)

print(f"✅ VectorStore 생성 완료!")
print(f"📊 저장된 문서 수: {vectorstore._collection.count()}")

In [ ]:
# 🔍 VectorStore 검색 테스트
print("🔍 VectorStore 유사도 검색 테스트")
print("=" * 60)

query = "LoRA 파인튜닝이란 무엇인가요?"
print(f"❓ 쿼리: '{query}'\n")

# 유사도 검색
results = vectorstore.similarity_search_with_score(query, k=3)

for i, (doc, score) in enumerate(results, 1):
    source = os.path.basename(doc.metadata.get('source', 'unknown'))
    print(f"📌 결과 {i} (유사도 점수: {score:.4f})")
    print(f"   출처: {source}")
    print(f"   내용: {doc.page_content[:120]}...")
    print()

## 5️⃣ LLM 설정 (OpenAI 또는 Ollama)

RAG 체인에서 사용할 LLM을 설정합니다.

In [ ]:
# 🤖 LLM 설정
if USE_OLLAMA:
    from langchain_community.llms import Ollama
    
    llm = Ollama(
        model="qwen2.5:1.5b",
        temperature=0.3,
    )
    print("✅ Ollama LLM 설정 완료 (모델: qwen2.5:1.5b)")
else:
    from langchain_openai import ChatOpenAI
    
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0.3,
    )
    print("✅ OpenAI LLM 설정 완료 (모델: gpt-4o-mini)")

# 간단한 테스트
print("\n🔧 LLM 테스트 중...")
try:
    test_response = llm.invoke("안녕하세요. 간단히 인사해주세요.")
    response_text = test_response if isinstance(test_response, str) else test_response.content
    print(f"💬 응답: {response_text[:100]}")
except Exception as e:
    print(f"❌ LLM 연결 실패: {e}")

## 6️⃣ RetrievalQA 체인 구축

LangChain의 **RetrievalQA**는 검색 + 생성을 하나의 체인으로 연결합니다.

### 🔗 체인 동작 흐름
```
사용자 질문 → Retriever(검색) → 관련 문서 추출 → 프롬프트 구성 → LLM 응답
```

In [ ]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# 📝 RAG 프롬프트 템플릿 정의
rag_prompt_template = """다음 컨텍스트를 참고하여 질문에 답변해주세요.
컨텍스트에 없는 정보는 '제공된 문서에서 해당 정보를 찾을 수 없습니다.'라고 답변하세요.

컨텍스트:
{context}

질문: {question}

답변:"""

PROMPT = PromptTemplate(
    template=rag_prompt_template,
    input_variables=["context", "question"]
)

print("📝 RAG 프롬프트 템플릿:")
print(rag_prompt_template)

In [ ]:
# 🔗 RetrievalQA 체인 생성
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},  # 상위 3개 문서 검색
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",  # 검색 결과를 하나로 합쳐서 전달
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT},
)

print("✅ RetrievalQA 체인 생성 완료")
print("  🔹 검색 방식: similarity")
print("  🔹 검색 문서 수: 3")
print("  🔹 체인 타입: stuff (모든 문서를 하나로)")

In [ ]:
# 🚀 RAG 질의응답 테스트
import time

def ask_rag(question):
    """RAG 체인에 질문하고 결과를 출력합니다."""
    print(f"❓ 질문: {question}")
    print("-" * 60)
    
    start_time = time.time()
    result = qa_chain.invoke({"query": question})
    elapsed = time.time() - start_time
    
    print(f"💬 답변:\n{result['result']}")
    print(f"\n⏱️ 소요 시간: {elapsed:.2f}초")
    
    print(f"\n📚 참고 문서:")
    for i, doc in enumerate(result['source_documents'], 1):
        source = os.path.basename(doc.metadata.get('source', 'unknown'))
        print(f"  {i}. [{source}] {doc.page_content[:80]}...")
    
    print("=" * 60)
    return result

# 테스트 질문
print("🚀 RAG 질의응답 테스트")
print("=" * 60)

ask_rag("LoRA와 QLoRA의 차이점은 무엇인가요?")

In [ ]:
# 📝 다양한 질문 테스트
questions = [
    "트랜스포머는 언제 발표되었나요?",
    "RAG 파이프라인의 주요 단계를 설명해주세요.",
    "딥러닝 시대는 어떻게 시작되었나요?",
    "양자 컴퓨터의 원리는 무엇인가요?",  # 문서에 없는 내용
]

print("📝 다양한 질문 테스트")
print("=" * 60)

for q in questions:
    print()
    ask_rag(q)

## 7️⃣ 대화형 RAG (Conversational RAG)

이전 대화 내용을 기억하면서 RAG를 수행하는 **대화형 RAG**를 구현합니다.

### 💡 일반 RAG vs 대화형 RAG
- 일반 RAG: 매번 독립적인 질문으로 처리
- 대화형 RAG: 이전 대화를 참고하여 "그것"과 같은 대명사 해석 가능

In [ ]:
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory

# 💾 대화 메모리 설정
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer",
)

# 🔗 대화형 RAG 체인 생성
conversational_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,
)

print("✅ 대화형 RAG 체인 생성 완료")
print("  🔹 대화 기록 유지: ConversationBufferMemory")
print("  🔹 이전 맥락을 참고하여 검색 쿼리 생성")

In [ ]:
# 💬 대화형 RAG 테스트
def chat_rag(question):
    """대화형 RAG에 질문합니다."""
    print(f"👤 사용자: {question}")
    
    result = conversational_chain.invoke({"question": question})
    answer = result["answer"] if isinstance(result["answer"], str) else result["answer"].content
    
    print(f"🤖 AI: {answer}")
    print("-" * 60)
    return result

print("💬 대화형 RAG 테스트")
print("=" * 60)

# 대화 1: 초기 질문
chat_rag("파인튜닝 방법에는 어떤 것들이 있나요?")
print()

# 대화 2: 대명사 사용 (이전 맥락 참조)
chat_rag("그 중에서 메모리를 가장 적게 사용하는 방법은?")
print()

# 대화 3: 후속 질문
chat_rag("그 방법으로 학습하려면 어떤 GPU가 필요한가요?")

## 8️⃣ LCEL (LangChain Expression Language)로 RAG 구현

최신 LangChain에서는 **LCEL**을 사용하여 더 유연하게 RAG 체인을 구성할 수 있습니다.

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 📝 LCEL 기반 RAG 체인
def format_docs(docs):
    """검색된 문서를 하나의 문자열로 포맷합니다."""
    return "\n\n".join(doc.page_content for doc in docs)

rag_prompt = PromptTemplate(
    template="""다음 컨텍스트를 참고하여 질문에 한국어로 답변해주세요.

컨텍스트:
{context}

질문: {question}

답변:""",
    input_variables=["context", "question"]
)

# LCEL 체인 구성
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("✅ LCEL 기반 RAG 체인 생성 완료")
print("  📊 구조: Retriever → Format → Prompt → LLM → Output")

# LCEL 체인 테스트
print("\n🚀 LCEL RAG 테스트")
print("=" * 50)

question = "RLHF와 DPO는 어떤 차이가 있나요?"
print(f"❓ 질문: {question}")

try:
    answer = rag_chain.invoke(question)
    response_text = answer if isinstance(answer, str) else answer.content
    print(f"\n💬 답변:\n{response_text}")
except Exception as e:
    print(f"❌ 에러: {e}")

In [ ]:
# 🧹 정리
import shutil

# 임시 파일 정리
if os.path.exists(sample_dir):
    shutil.rmtree(sample_dir)
    print(f"🧹 임시 디렉토리 삭제: {sample_dir}")

print("\n📌 오늘의 핵심 정리:")
print("  1️⃣ LangChain Document Loaders로 다양한 문서 형식을 로딩")
print("  2️⃣ Text Splitters로 문서를 적절한 크기로 분할")
print("  3️⃣ HuggingFace Embeddings + ChromaDB로 벡터 검색")
print("  4️⃣ RetrievalQA로 검색+생성을 하나의 체인으로 구현")
print("  5️⃣ 대화형 RAG로 이전 맥락을 유지하며 대화")
print("  6️⃣ LCEL로 더 유연한 RAG 체인 구성 가능")

---

## 9️⃣ Streamlit으로 RAG 챗봇 서비스 만들기

§1~§8에서 만든 RAG 부품들(Loader · Splitter · VectorStore · LLM · LCEL)을 **실제 사용자가 쓰는 웹 챗봇 서비스**로 만들어봅시다.

### 🎯 목표

**PDF Q&A 챗봇** — PDF를 업로드하고 자연어로 질문하면 답변 + 출처를 보여주는 웹 앱.

### 📐 앱 아키텍처

```
┌─────────────────────────┐    ┌──────────────────────────┐
│  Streamlit UI           │    │  Python Backend          │
│  ─ 사이드바 (업로드/설정)│ ←→ │  ─ 인덱싱 (PyPDF→Chroma) │
│  ─ 메인 (채팅 이력)      │    │  ─ 질의 (LCEL 체인)      │
│  ─ 입력 (chat_input)    │    │  ─ 스트리밍 (chain.stream)│
└─────────────────────────┘    └──────────────────────────┘
```

### 📦 필요 패키지 (이미 설치돼있음 + streamlit)

```
streamlit  langchain  langchain-openai  langchain-chroma
langchain-community  chromadb  pypdf  python-dotenv
```

### 🔧 단계
1. 셀 ①: `app.py` 작성 (`%%writefile`)
2. 셀 ②: `requirements.txt` 작성
3. 셀 ③: `.env.example` 작성
4. 셀 ④: 터미널에서 `streamlit run app.py` 실행 안내

> 💡 노트북 셀에서 Streamlit 앱을 직접 실행할 수는 없습니다 (Streamlit은 별도 프로세스). 노트북은 **파일을 만드는 용도**, 실행은 터미널에서 합니다.

---

### ① `app.py` 작성

아래 셀을 실행하면 같은 폴더에 `streamlit_app/app.py` 가 만들어집니다.

In [ ]:
%%writefile streamlit_app/app.py
"""
PDF RAG 챗봇 (Streamlit) — Session 28 실습
============================================
사용법:
    1) cp .env.example .env  → OPENAI_API_KEY 입력
    2) pip install -r requirements.txt
    3) streamlit run app.py

기능:
    - PDF 여러 개 업로드 → 자동 인덱싱 (Chroma)
    - 대화 이력 유지 (멀티턴)
    - 스트리밍 응답 + 출처 표시
    - 모델·청크 크기 사이드바에서 조절
    - 대화 초기화 / DB 리셋 버튼
"""
from __future__ import annotations

import os
import tempfile
from pathlib import Path

import streamlit as st
from dotenv import load_dotenv

# === LangChain 임포트 ============================================
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document


# === 0. 환경 설정 ================================================
load_dotenv()

st.set_page_config(
    page_title="📚 PDF RAG 챗봇",
    page_icon="📚",
    layout="wide",
    initial_sidebar_state="expanded",
)


# === 1. 헬퍼 함수 =================================================
def get_api_key() -> str | None:
    """API 키를 환경변수 또는 Streamlit secrets에서 가져옴."""
    key = os.getenv("OPENAI_API_KEY")
    if not key:
        try:
            key = st.secrets.get("OPENAI_API_KEY")
        except Exception:
            key = None
    if key and key.startswith("sk-your"):
        return None  # placeholder
    return key


@st.cache_resource(show_spinner="📚 PDF 인덱싱 중... (최초 1회만)")
def build_vectorstore(
    file_bytes_list: list[tuple[str, bytes]],
    chunk_size: int = 500,
    chunk_overlap: int = 50,
) -> Chroma | None:
    """업로드된 PDF 바이트들을 인덱싱하여 Chroma 반환.

    Streamlit @st.cache_resource로 같은 입력엔 1회만 실행.
    file_bytes_list: [(filename, bytes), ...] — 캐시 키로 쓰임
    """
    if not file_bytes_list:
        return None

    # 1) PDF 로드 (임시 파일 경유 — PyPDFLoader는 경로 필요)
    all_docs: list[Document] = []
    for filename, file_bytes in file_bytes_list:
        with tempfile.NamedTemporaryFile(
            delete=False, suffix=".pdf"
        ) as tmp:
            tmp.write(file_bytes)
            tmp_path = tmp.name
        try:
            pages = PyPDFLoader(tmp_path).load()
            # 메타데이터에 원본 파일명 주입
            for p in pages:
                p.metadata["source"] = filename
            all_docs.extend(pages)
        finally:
            os.unlink(tmp_path)

    # 2) 청킹
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(all_docs)

    # 3) Chroma — in-memory (앱 재시작 시 휘발)
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
        collection_name="pdf_rag_session",
    )
    return vectorstore


def build_rag_chain(vectorstore: Chroma, model_name: str, top_k: int):
    """RAG LCEL 체인 — retriever | prompt | llm | parser."""
    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": top_k},
    )
    llm = ChatOpenAI(model=model_name, temperature=0, streaming=True)

    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "당신은 PDF 문서를 참고해 정확하게 답하는 어시스턴트입니다.\n"
         "주어진 문서에서 근거를 찾아 답하세요.\n"
         "문서에 없는 내용은 \"제공된 문서에서 찾을 수 없습니다\" 라고 답하세요.\n"
         "한국어로 답하세요."),
        ("human",
         "참고 문서:\n{context}\n\n질문: {question}"),
    ])

    def format_docs(docs: list[Document]) -> str:
        return "\n\n".join(
            f"[문서 {i+1}] {d.page_content}"
            for i, d in enumerate(docs)
        )

    chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough(),
        }
        | prompt
        | llm
        | StrOutputParser()
    )
    return chain, retriever


def render_sources_expander(retrieved_docs: list[Document]) -> list[str]:
    """검색된 문서를 expander로 표시하고 인용 문자열 반환."""
    citations: list[str] = []
    with st.expander(f"📚 출처 보기 (Top-{len(retrieved_docs)})"):
        for i, doc in enumerate(retrieved_docs, 1):
            src = doc.metadata.get("source", "?")
            page = doc.metadata.get("page", "?")
            preview = doc.page_content[:250].replace("\n", " ")
            st.markdown(f"**[{i}]** `{src}` · p.{page}")
            st.caption(preview + ("..." if len(doc.page_content) > 250 else ""))
            citations.append(f"{src} p.{page}")
    return citations


# === 2. UI ========================================================
st.title("📚 PDF RAG 챗봇")
st.caption("PDF를 업로드하고 자연어로 질문하세요. LangChain + OpenAI + Chroma 기반.")

# --- 사이드바 ---
with st.sidebar:
    st.header("⚙️ 설정")

    # API 키 확인
    api_key = get_api_key()
    if not api_key:
        st.error("❌ OPENAI_API_KEY가 설정되지 않았습니다.")
        st.markdown(
            ".env 파일에 `OPENAI_API_KEY=sk-...` 추가 또는\n"
            "Streamlit Cloud의 Secrets에 등록해주세요."
        )
        st.stop()
    os.environ["OPENAI_API_KEY"] = api_key
    st.success(f"✅ API 키 OK (`{api_key[:8]}...`)")

    st.divider()

    # PDF 업로드
    st.subheader("📄 PDF 업로드")
    uploaded_files = st.file_uploader(
        "PDF 파일을 선택하세요",
        type=["pdf"],
        accept_multiple_files=True,
        help="여러 PDF를 동시에 업로드할 수 있습니다",
    )

    st.divider()

    # 모델 / 검색 설정
    st.subheader("🛠️ 모델 / 검색")
    model_name = st.selectbox(
        "LLM 모델",
        ["gpt-4o-mini", "gpt-4o", "gpt-4.1-mini"],
        index=0,
        help="gpt-4o-mini가 가장 저렴 (Top 추천)",
    )
    chunk_size = st.slider("청크 크기 (자)", 200, 2000, 500, 100)
    chunk_overlap = st.slider("청크 오버랩 (자)", 0, 200, 50, 10)
    top_k = st.slider("검색 Top-K", 1, 10, 3)

    st.divider()

    # 컨트롤
    st.subheader("🔧 컨트롤")
    if st.button("🗑️ 대화 초기화"):
        st.session_state.messages = []
        st.rerun()
    if st.button("♻️ 인덱스 재구축"):
        st.cache_resource.clear()
        st.session_state.messages = []
        st.rerun()

    st.divider()
    st.caption("💡 Tip: 청크 크기/Top-K를 바꾸면 답변 품질이 달라집니다.")


# --- 메인: 인덱싱 + 채팅 ---

# 인덱싱 — 업로드된 파일 바이트로 vectorstore 만들기
vectorstore: Chroma | None = None
if uploaded_files:
    # bytes 추출 (캐시 키용 — 파일명과 바이트가 같으면 캐시 히트)
    file_bytes = [(f.name, f.getvalue()) for f in uploaded_files]
    vectorstore = build_vectorstore(file_bytes, chunk_size, chunk_overlap)
    if vectorstore:
        n_docs = vectorstore._collection.count()
        st.sidebar.success(f"✅ 인덱스 준비 완료 — 청크 {n_docs}개")

# 채팅 영역
if not vectorstore:
    st.info("👈 왼쪽 사이드바에서 PDF를 먼저 업로드해주세요.")
    st.stop()

# 세션 상태 초기화
if "messages" not in st.session_state:
    st.session_state.messages = []

# 기존 대화 이력 표시
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])
        if msg.get("sources"):
            with st.expander("📚 출처"):
                for src in msg["sources"]:
                    st.markdown(f"- {src}")

# 사용자 입력 + 응답
if user_q := st.chat_input("PDF에 대해 물어보세요…"):
    # 1) 사용자 메시지 저장 + 표시
    st.session_state.messages.append({
        "role": "user",
        "content": user_q,
    })
    with st.chat_message("user"):
        st.markdown(user_q)

    # 2) 어시스턴트 답변 — 스트리밍
    with st.chat_message("assistant"):
        # 체인 + retriever 생성
        chain, retriever = build_rag_chain(
            vectorstore, model_name, top_k
        )

        # 먼저 검색해서 출처 확보 (스트리밍 전에)
        retrieved = retriever.invoke(user_q)

        # 스트리밍 응답 — st.write_stream이 generator를 받아 토큰 단위 렌더링
        try:
            answer = st.write_stream(chain.stream(user_q))
        except Exception as e:
            st.error(f"❌ 오류: {e}")
            answer = ""

        # 출처 표시
        citations = render_sources_expander(retrieved)

        # 메시지 저장
        st.session_state.messages.append({
            "role": "assistant",
            "content": answer,
            "sources": citations,
        })


### ② `requirements.txt` 작성

In [ ]:
%%writefile streamlit_app/requirements.txt
streamlit>=1.30.0
langchain>=0.2.0
langchain-openai>=0.1.0
langchain-chroma>=0.1.0
langchain-community>=0.2.0
langchain-core>=0.2.0
chromadb>=0.4.22
pypdf>=4.0.0
python-dotenv>=1.0.0


### ③ `.env.example` 작성

> ⚠️ 절대 실제 키를 `.env.example` 에 넣지 마세요. 실행 전 `cp .env.example .env` 로 복사 후 실제 키 입력.

In [ ]:
%%writefile streamlit_app/.env.example
# OpenAI API Key — https://platform.openai.com/api-keys
OPENAI_API_KEY=sk-your-api-key-here


### ④ 로컬 실행

터미널을 새로 열고 다음 명령을 차례로 실행합니다.

In [ ]:
# 실행 명령 출력 (복사해서 터미널에서 사용)
print(r'''
# 1) 폴더 이동
cd streamlit_app

# 2) (선택) 가상환경 — 기존 venv 활용하면 스킵
python -m venv venv && source venv/bin/activate   # Windows: venv\Scripts\activate

# 3) 패키지 설치
pip install -r requirements.txt

# 4) API 키 설정
cp .env.example .env          # .env 파일 열고 OPENAI_API_KEY=sk-... 실제 키 입력

# 5) 실행
streamlit run app.py

# → 브라우저 자동 오픈 http://localhost:8501
''')

### ⑤ 사용 흐름

브라우저가 열리면:

1. **사이드바 → PDF 업로드** (여러 개 가능)
2. 인덱싱 완료 메시지 확인 (`✅ 인덱스 준비 완료 — 청크 N개`)
3. 사이드바에서 **모델·청크 크기·Top-K** 조절 (선택)
4. 하단 채팅창에 **자연어 질문** 입력
5. 답변이 스트리밍으로 표시되고 아래 `📚 출처 보기` 클릭으로 근거 확인

### 흔한 문제 & 해결

| 증상 | 해결 |
|------|------|
| `401 Authentication Error` | `.env`의 키가 placeholder인지 확인 (sk-your-... → 실제 키로) |
| `Quota exceeded` | OpenAI billing 충전 또는 `gpt-4o-mini` 사용 |
| 답변이 엉뚱함 | 사이드바에서 청크 크기 / Top-K 조절 |
| 앱이 느림 | 인덱싱 캐시(`@st.cache_resource`)가 적용됐는지 확인 |
| `ModuleNotFoundError: streamlit` | `pip install -r requirements.txt` 다시 실행 |

### 🚀 Streamlit Cloud 배포 (선택)

1. 코드를 GitHub에 push (`.env` 제외 — `.gitignore` 사용)
2. https://share.streamlit.io 접속 → GitHub 연동
3. **New app** → repo 선택 → `streamlit_app/app.py` 경로 지정
4. **Secrets**에 `OPENAI_API_KEY = "sk-..."` 입력
5. Deploy → 공개 URL 발급 (예: `https://my-pdf-rag.streamlit.app`)

---

---

## 🎯 실습 과제

1️⃣ 자신만의 문서(PDF 또는 TXT)를 로딩하여 RAG 시스템을 구축해보세요  
2️⃣ 프롬프트 템플릿을 수정하여 응답 품질을 개선해보세요  
3️⃣ `search_kwargs`의 `k` 값을 1, 3, 5로 바꿔 결과를 비교해보세요  

---

## 📚 참고 자료
- [LangChain 공식 문서](https://python.langchain.com/docs/)
- [LangChain RAG 튜토리얼](https://python.langchain.com/docs/tutorials/rag/)
- [LCEL 가이드](https://python.langchain.com/docs/concepts/lcel/)

### 🌐 Streamlit 챗봇 실습 (§9 연계)

5. **본인 도메인 PDF 1~3개** 를 streamlit_app에 업로드하고 5개 이상 질문해보기
6. 사이드바에서 **청크 크기 256 / 512 / 1024** 으로 바꾸며 답변 차이 관찰
7. (선택) **GitHub push → Streamlit Cloud** 무료 배포 — 공개 URL 만들기